> ⚠️ **Before you start:** This is a read-only course copy.
> Go to **File → Save a copy in Drive** right now, then continue working in *your* copy.
> Changes made here will not be saved.

In [ ]:
# Setup: install the course package (run this once per Colab session).
#
# Why the uninstall: the package version number never changes, so on a runtime
# that already has a copy, a plain install reports "already satisfied", skips the
# download, and leaves you running whatever code you installed last time.
#
# If you already ran an import cell before this one, use Runtime > Restart session
# afterwards. Python keeps the old module in memory even once the files are new.
!pip uninstall -q -y churn-pipeline
!pip install -q git+https://github.com/marceloacosta/churn-prediction-pipeline.git@main

# Chapter 7: LLM Integration

## Using Claude to Do the Boring Parts of ML Ops

Our pipeline has two places where a task is fundamentally about *understanding language*, something humans do effortlessly and rule-based code struggles with:

1. **Auto-Mapping (onboarding):** A new client uploads a CSV. Someone has to figure out what `MonthlyCharges`, `mrr`, or `amt_per_month` all mean. That's pattern recognition on natural language.

2. **Narrative Generation (output):** SHAP gives us `contract_type=month-to-month (+0.23)`. A business user needs: "This customer has no long-term commitment." That's translation from numbers to English.

Neither task is new to you. The telco mapping you wrote by hand in Chapter 1 has been translating that client's columns in every chapter since, and you spent Chapter 4 reading SHAP output yourself. What this chapter automates is the second client, and the thousandth: the LLM drafts what you once wrote by hand, and reviewing its draft takes exactly the judgement you built by writing one. That is why the by-hand version came six chapters before this one.

Both tasks call Claude on Amazon Bedrock. A failed call raises `BedrockCallError` with the cause in the message; what happens after that differs by step, and the failure section at the end covers both.

They are not equally harmless, though, and it is worth having the difference straight before you start. A missing narrative is cosmetic. The column reads `"N/A"` and the scores, risk tiers and SHAP reasons all ship as normal. A missing mapping stops that client dead: there is nothing to translate their columns with, and `load_mapping_config` raises on the file that is not there. Onboarding waits for a human.

So only one of these two is really optional. **What happens when it fails**, near the end, works through both, including the failure that costs the most: a confidently wrong answer.

### Cost

| Task | Typical Cost | When It Runs |
|------|-------------|-------------|
| Auto-Mapping | ~$0.01 per new client | Once, during onboarding |
| Narratives (50 customers) | ~$0.01-0.02 per batch | Each scoring run |

### Where the code lives

The pip cell above installed the course package from the repo, which makes
`churn_pipeline` importable here like any other library. This chapter drives two files
inside it:

- **`churn_pipeline/llm/auto_mapping.py`**, behind Part 1. It turns a client's column
  names into a mapping config a human can review.
- **`churn_pipeline/llm/narrative_generator.py`**, behind Part 2. It turns SHAP output
  into paragraphs a business user can act on.

Both files are organised the same way, as three plain functions you could have written
yourself: one **builds** a prompt string out of your data, one **sends** that string to
Bedrock and returns the reply, and one **parses** the reply into Python objects. That is
the entire trick of the chapter. The model only ever sees a string, so all of the
engineering is in what goes into that string and what you do with the one that comes
back.

To read any of these functions right here, `import inspect` and print, for example,
`inspect.getsource(build_mapping_prompt)`. The remaining imports below are the approval
machinery from Chapter 1, which decides whether the pipeline may act on what the model
produced.

In [ ]:
# --- Part 1: auto-mapping. Build a prompt from a client's columns, send it, and
# turn the reply into a draft YAML somebody can review.
from churn_pipeline.llm.auto_mapping import (
    build_mapping_prompt,     # columns + sample rows + the standard schema -> one prompt
    call_bedrock_for_mapping, # sends it and parses the reply; raises BedrockCallError on failure
    write_draft_yaml,         # writes the suggestions out as mapping.draft.yaml
    is_mapping_approved,      # cheap check: is there an approved mapping.yaml here?
    ColumnMapping,            # one suggestion: source column, target field, confidence, why
    _parse_mapping_response,  # Claude's JSON -> ColumnMapping objects. Underscore because
                              # it is internal; we call it directly to parse a saved reply.
)

# --- What every failed Bedrock call raises. The message names the cause, which is
# how the two steps above can react differently to the same kind of failure.
from churn_pipeline.llm.bedrock import BedrockCallError

# --- Part 2: narratives. Turn SHAP numbers for a batch of customers into English.
from churn_pipeline.llm.narrative_generator import (
    build_narrative_prompt,         # a batch of customers + their SHAP features -> one prompt
    call_bedrock_for_narratives,    # sends it; returns {customer_id: text}; raises on failure
    parse_narrative_response,       # splits one reply back into per-customer narratives
    generate_narratives_for_batch,  # batching, per-customer failure reasons, the budget
    NarrativeGenerationError,       # raised when a run fails more customers than its budget
    NarrativeRequest,               # one customer's prediction going in
    SYSTEM_PROMPT,                  # the house style every narrative has to follow
)

# --- The approval gate, from the config format built in chapter 1.
from churn_pipeline.mapping_config import (
    load_mapping_config,      # reads an approved mapping.yaml
    MappingNotApprovedError,  # ...and what it raises instead when handed a draft
)

# --- The standard field names every client's columns get translated into. This is
# the target vocabulary we hand the LLM.
from churn_pipeline.data_contract import STANDARD_SCHEMA

## First: are we calling Bedrock for real?

This chapter can run two ways.

**With credentials**, every prompt below goes to Claude on Amazon Bedrock and you
see what actually comes back, which is the point, because what comes back is not
identical every time.

**Without credentials**, the notebook falls back to saved responses so you can keep
reading: the failed call raises `BedrockCallError`, the cell catches it, says so, and
substitutes a saved reply. Production code never makes that substitution, because a
canned narrative is indistinguishable from a real one by the time it reaches a client;
the last section of this chapter covers what it does instead.

If you want the live version, work through the **AWS credentials** setup page first:
one Bedrock API key in Colab Secrets, about five minutes. Otherwise just run on.

In [ ]:
import os


def load_aws_secrets():
    """Copy Colab Secrets into the environment, where boto3 looks for them.

    Returns the secrets it could not read, with the reason. Missing secrets are
    not an error here; the cells below fall back to saved responses.
    """
    try:
        from google.colab import userdata
    except ImportError:
        return  # not in Colab: use whatever credentials this machine has

    missing = []
    for name in ("AWS_BEARER_TOKEN_BEDROCK", "AWS_DEFAULT_REGION"):
        try:
            os.environ[name] = userdata.get(name)
        except Exception as e:
            missing.append((name, type(e).__name__))
    return missing


missing = load_aws_secrets() or []

LIVE = bool(os.environ.get("AWS_BEARER_TOKEN_BEDROCK"))

if LIVE:
    print(f"Credentials found. Calling Bedrock for real in {os.environ.get('AWS_DEFAULT_REGION')}.")
elif any(reason == "NotebookAccessError" for _, reason in missing):
    # The secrets exist but this notebook is not allowed to read them. Every
    # notebook needs its own toggle, including copies you saved to Drive.
    print("Your secrets exist, but this notebook cannot read them.")
    print("Open the Secrets panel (key icon) and switch Notebook access on for:")
    for name, reason in missing:
        if reason == "NotebookAccessError":
            print(f"  - {name}")
    print("Then run this cell again. Falling back to saved responses for now.")
else:
    print("No credentials found. Running on saved responses.")
    print("(A notebook stand-in so you can keep reading. What production does")
    print(" instead is the 'What happens when it fails' section near the end.)")

## Part 1: Auto-Mapping, or Teaching a Machine to Read Column Names

### The Problem

Every company calls their data something different:

| Company A | Company B | Company C | They All Mean... |
|-----------|-----------|-----------|------------------|
| MonthlyCharges | mrr | monthly_fee | monthly_charges |
| customerID | CustID | user_id | customer_id |
| Churn | left_service | is_churned | churn_label |

A rule-based approach would need an infinite dictionary of synonyms. An LLM handles this naturally because it *understands language*.

In [ ]:
# A new client's export. These eight columns are picked to cover the three cases you
# actually meet, because a sample where everything is obvious teaches you nothing:
#
#   easy       CustID, MonthlyFee, TotalSpend   -> near-synonyms of our field names
#   reworded   months_active, left_service      -> same meaning, no shared words
#   arguable   complaints, how_they_pay         -> a judgement call, and the place
#                                                  where a reviewer earns their keep
#
# Two rows is enough. The model needs values to disambiguate names, not statistics.
client_columns = ["CustID", "months_active", "MonthlyFee", "TotalSpend",
                   "left_service", "plan_type", "how_they_pay", "complaints"]

sample_rows = [
    {"CustID": "USR-7590", "months_active": 12, "MonthlyFee": 59.99,
     "TotalSpend": 719.88, "left_service": "no", "plan_type": "annual",
     "how_they_pay": "credit card", "complaints": 0},
    {"CustID": "USR-3344", "months_active": 2, "MonthlyFee": 99.99,
     "TotalSpend": 199.98, "left_service": "yes", "plan_type": "monthly",
     "how_they_pay": "bank transfer", "complaints": 4},
]

print("Client's raw columns:", client_columns)
print(f"\nSample row: {sample_rows[0]}")

### What goes into the prompt

`build_mapping_prompt` is a plain Python function in
`churn_pipeline/llm/auto_mapping.py`, installed by the pip cell at the top; run
`inspect.getsource(build_mapping_prompt)` if you want to read it here. It is where the
actual engineering of this step lives, and everything else in Part 1 is plumbing around
it. It assembles four things, and each one is doing a job:

**1. The client's column names.** What we are mapping from.

**2. Two rows of their data.** The part people skip, and the part that earns its place.
A column called `id` could be anything. A column called `id` holding `USR-7590` is a
customer identifier and the model can see that. Names alone are ambiguous; names plus
values usually are not.

**3. The standard fields, with descriptions.** The vocabulary we are mapping *to*, read
straight out of `STANDARD_SCHEMA` in the data contract from Chapter 1. Notice the
descriptions carry real domain knowledge, like "month-to-month customers leave 3x more
often". That is there for the model, not for us. And because it is generated from the
contract, adding a field to the contract updates this prompt automatically. There is no
second place to remember.

**4. The output contract.** Return a JSON array, four keys per entry, `null` for
anything you cannot place. Then the line that matters most:

> *"Only map columns where you have reasonable confidence. It's better to leave a column
> unmapped than to guess wrong."*

That is a deliberate push toward saying nothing. The whole argument of this chapter is
that a confident wrong mapping is the expensive failure, so the prompt is written to
make the model abstain when it is unsure. An unmapped column is a question for a human.
A wrongly mapped one is a bug you find in three months.

One limitation worth knowing: the prompt includes at most **3 sample rows** and the
first **8 columns of each**. A client with 40 columns gets all 40 names listed, but
values for only the first 8, so the disambiguation in point 2 quietly stops helping
past that point.

In [ ]:
prompt = build_mapping_prompt(client_columns, sample_rows)

# Printed in full on purpose. This is the entire input Claude sees, and there is
# nothing else: no examples, no conversation history, no hidden preamble.
print(f"The complete prompt, {len(prompt)} characters:")
print("=" * 70)
print(prompt)

In [ ]:
# The real call. call_bedrock_for_mapping sends the prompt and parses the JSON reply
# into ColumnMapping objects. On any failure (bad key, throttling, a reply that is not
# JSON) it raises BedrockCallError with the cause in the message. The try/except below
# is the notebook's fallback so you can read on without credentials.
SAVED_MAPPING_RESPONSE = """[
    {"source_column": "CustID", "target_field": "customer_id", "confidence": "high", "reasoning": "Contains 'ID' and values look like unique identifiers"},
    {"source_column": "months_active", "target_field": "tenure_months", "confidence": "high", "reasoning": "Directly describes duration in months"},
    {"source_column": "MonthlyFee", "target_field": "monthly_charges", "confidence": "high", "reasoning": "Monthly + Fee = monthly billing amount"},
    {"source_column": "TotalSpend", "target_field": "total_charges", "confidence": "high", "reasoning": "Cumulative spending"},
    {"source_column": "left_service", "target_field": "churn_label", "confidence": "medium", "reasoning": "Binary indicator of leaving, needs value mapping yes/no to 1/0"},
    {"source_column": "plan_type", "target_field": "contract_type", "confidence": "medium", "reasoning": "Describes contract duration category"},
    {"source_column": "how_they_pay", "target_field": "payment_method", "confidence": "high", "reasoning": "Payment method description"},
    {"source_column": "complaints", "target_field": "support_tickets", "confidence": "medium", "reasoning": "Complaint count likely correlates with support interactions"}
]"""

mappings = None
if LIVE:
    try:
        mappings = call_bedrock_for_mapping(prompt)
        print(f"Live from Bedrock: {len(mappings)} columns mapped.\n")
    except BedrockCallError as e:
        print(f"Bedrock call failed: {e}")
        print("To work out what to do about it, run the verify cell on the AWS")
        print("credentials page. Using the saved response so you can keep reading.\n")

if mappings is None:
    mappings = _parse_mapping_response(SAVED_MAPPING_RESPONSE)

# `confidence` is the model's own grade, and it is the reviewer's reading order rather
# than a number to threshold on. `high` means the names line up. `medium` means it made
# a defensible guess from the sample values. Nothing in the pipeline treats them
# differently; a person does.
print("Claude's mapping suggestions:")
print("=" * 70)
print(f"{'Client Column':<18} {'\u2192 Standard Field':<20} {'Confidence':<12} Reasoning")
print("-" * 70)
for m in mappings:
    print(f"{m.source_column:<18} \u2192 {m.target_field:<18} {m.confidence:<12} {m.reasoning[:40]}")
# The prompt tells the model to return target_field: null rather than guess, and
# _parse_mapping_response drops those entries. Dropping them quietly would hide the
# abstention we asked for, so check what did not survive.
unmapped = set(client_columns) - {m.source_column for m in mappings}
print(f"\nColumns Claude declined to map: {sorted(unmapped) or 'none this time'}")
print("Anything listed there is a question for the reviewer, not a failure.")


### The Approval Workflow

The LLM's output is a **draft**. Three things have to happen before the pipeline will
touch it, and the next three cells do them one at a time.

**Why is a rename the approval?** Because the gate has to be something a person does on
purpose and anyone can check. No database, no approvals table, no state that can drift
out of sync with the file. You can answer "is this approved?" with `ls`.

The code enforces this. `load_mapping_config()` refuses a draft and raises
`MappingNotApprovedError`, and there is no flag to skip it. It checks two things,
because either one alone is easy to defeat by accident: the filename, and whether the
file still says `status: draft` inside.

#### Step 1: the LLM writes a draft

`write_draft_yaml` turns the suggestions into a file. Two things in it are the reason
this step exists at all:

- **`status: draft`** and the `.draft.yaml` filename. Both mark it unreviewed.
- **`confidence_scores`.** The model's own opinion of each guess, so a reviewer knows
  where to look first. `medium` is where the mistakes live.

Read the output. That file is what a human is being asked to check.

In [ ]:
import os

import yaml

# A real folder under the working directory. In Colab: the folder icon in the left
# sidebar, then client_configs/new_client.
config_dir = "client_configs/new_client"
os.makedirs(config_dir, exist_ok=True)
draft_path = os.path.join(config_dir, "mapping.draft.yaml")
approved_path = os.path.join(config_dir, "mapping.yaml")

write_draft_yaml(mappings, "new_client", draft_path)

print(f"The LLM wrote {draft_path}:")
print("-" * 64)
print(open(draft_path).read())

# And nothing downstream will touch it.
try:
    load_mapping_config(draft_path)
except MappingNotApprovedError as e:
    print(f"The pipeline refuses to load it:\n  {e}")

#### Step 2: a human reviews it

This is the step the whole chapter is built around, and in real life it is a person
opening the file in an editor.

Look at `value_mappings` in the output above: it is empty. Claude could see that
`left_service` holds `"yes"` and `"no"`, but nothing in the CSV tells it the pipeline
stores churn as `1` and `0`. The model had no way to avoid that gap, and it did not
flag it either, because the column mapping itself was right. Only somebody who knows
the contract can catch it.

That is the shape of this review in general: the model gets you most of the way and
cannot know what it cannot see. The cell below makes the edit in code so the chapter
can carry on. Nothing about it is automatic.

In [ ]:
config = yaml.safe_load(open(draft_path))

# The two edits a reviewer makes: fill in what the model could not know, then say out
# loud that a person has looked at it.
config["value_mappings"] = {"churn_label": {"yes": 1, "no": 0}}
config["status"] = "approved"

with open(draft_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)

print("Two keys changed:")
print(f"  status:         'draft' -> {config['status']!r}")
print(f"  value_mappings: {{}} -> {config['value_mappings']}")
print("\nStill named .draft.yaml, so the pipeline still will not load it.")

#### Step 3: the rename is the approval

The file is correct now, and the pipeline still refuses it, because being correct and
being approved are two different claims. The rename is where a person makes the second
one.

Watch the three checks below. The draft does not count even after it was fixed;
`mapping.yaml` does not exist until the rename; and the moment it does, the gate opens.

In [ ]:
print("Before the rename:")
print(f"  mapping.draft.yaml approved? {is_mapping_approved(draft_path)}   (a draft never counts)")
print(f"  mapping.yaml approved?       {is_mapping_approved(approved_path)}   (does not exist yet)")

os.rename(draft_path, approved_path)

print("\nAfter the rename:")
print(f"  mapping.yaml approved?       {is_mapping_approved(approved_path)}")

loaded = load_mapping_config(approved_path)
print(f"\nload_mapping_config accepts it: client_id={loaded.client_id!r}, "
      f"{len(loaded.column_mappings)} columns mapped.")

# Renaming is not a way around reading. Someone who skipped step 2 would still have
# status: draft in the file. Shown on a throwaway copy so the real one stays approved.
lazy_path = os.path.join(config_dir, "renamed_but_unread.yaml")
config["status"] = "draft"
with open(lazy_path, "w") as f:
    yaml.dump(config, f, default_flow_style=False, sort_keys=False)
try:
    load_mapping_config(lazy_path)
except MappingNotApprovedError as e:
    print(f"\nSkipping step 2 and renaming anyway does not work:\n  {e}")
os.remove(lazy_path)

print("\nOpen the folder icon in the Colab sidebar and you will find one file:")
print(f"  {approved_path}")
print("No mapping.draft.yaml beside it: step 3 renamed the draft rather than copying")
print("it, which is what makes 'is there a mapping.yaml here' a complete answer.")

### What Part 1 just did

| Step | What happened | What came out |
|------|---------------|---------------|
| Sample CSV | A new client's column names, plus two rows of data | `client_columns`, `sample_rows` |
| Build prompt | Wrapped those in instructions and the standard schema | a prompt of about 2,000 characters |
| Call Claude | Sent it to Bedrock, or used the saved reply | 8 `ColumnMapping` objects |
| Draft, review, rename | Wrote `mapping.draft.yaml`, filled in `value_mappings`, renamed it | `client_configs/new_client/mapping.yaml` |

**That last file is the actual output.** Everything the pipeline does for this client
from here on reads it to translate their column names into the standard schema. In
Chapter 1 you wrote a file like it by hand. What changed is who drafted it, and a human
still signs off before it counts.

### So when does scoring happen?

Not now. Approving the mapping is the end of onboarding, and onboarding happens once
per client. Nothing gets scored at that moment.

Scoring happens on every run. A run is the chapter 1 to 4 chain executed against a
fresh export from the client, and the approved `mapping.yaml` is the first thing it
touches:

1. Load `mapping.yaml` and apply it: the client's raw CSV goes in, standard columns
   come out (Chapter 1).
2. Clean and validate the renamed data (Chapter 2).
3. Build features (Chapter 3).
4. The model scores every customer and SHAP explains every score (Chapter 4). You did
   this for real there: 7,043 Telco customers in, `telco_predictions.csv` out.

The output of step 4 is the scored table, one row per customer with a probability, a
risk tier and the top SHAP features. Part 2 below picks up exactly there and writes
narratives for the customers worth calling.

So over the life of one client: approve the mapping once, then map, score and narrate
on every run, reusing the same file until the client changes their export format. This
notebook does not re-run chapters 1 to 4; Part 2 uses two hand-made rows of the scored
table so the chapter stands alone.

## Part 2: Narrative Generation, or SHAP Numbers Turned Into English

### The Problem

After scoring, we have:
```
contract_type=month-to-month (+0.23); tenure_months=2 (+0.15); support_tickets=5 (+0.18)
```

A data scientist reads that and understands. A VP of Customer Success needs:

> "This customer has no long-term commitment and has contacted support 5 times in just 2 months. Month-to-month customers with high support volume are among the most likely to leave. Consider offering a discounted annual plan."

The LLM does this translation at scale.

### Where a batch comes from, and why it is a batch

**Who gets a narrative?** In Chapter 4 the pipeline scored every customer and gave each
one a probability and a tier: **high** at 0.70 and up, **medium** from 0.40, **low**
below that. That scored table is the input here.

Not everybody in it. The two customers below sit at 0.87 and 0.72, both high risk, and
those are the people somebody is going to phone this week. A paragraph about a customer
at 3% risk costs the same to generate and nobody will open it. On a 7,000-row file the
high tier is usually a few hundred rows, so filtering first is most of your cost saving
before you have tuned anything else.

**Why send them together?** The instructions, the feature definitions and the system
prompt are identical for every customer. Batch 50 into one call and you pay for that
preamble once. Make 50 separate calls and you pay for it 50 times, and wait through 50
round trips instead of one.

**Why 50 and not 500?** Three reasons, all pulling the same direction:

- A failed batch takes every customer in it down. At 50, an outage costs you 50
  narratives. At 500 it costs 500.
- Long prompts drift. Ask for 500 paragraphs in one go and the later ones get terse.
- The whole reply has to fit inside the output limit, and 50 paragraphs at 150 words
  each is already a large response.

None of this is managed by hand. `generate_narratives_for_batch` slices whatever list
you give it into groups of `batch_size`, 50 unless you pass something else, and makes
one call per group, so 400 high-risk customers means eight calls. The two customers
below go through exactly that machinery as one small batch; there are two of them so
that you can read the whole prompt on screen.

In [ ]:
# Two customers, standing in for the high-risk slice of a scored run. Each carries what
# the model needs in order to write about it: how likely it is to leave, and which
# features pushed it there. `contribution` is the SHAP value from Chapter 4, meaning how
# much that feature moved this particular prediction, not how important it is overall.
batch = [
    NarrativeRequest(
        customer_id="CUST_001",
        churn_probability=0.87,
        risk_tier="high",
        top_shap_features=[
            {"feature": "contract_type", "contribution": 0.23},
            {"feature": "support_tickets", "contribution": 0.18},
            {"feature": "tenure_months", "contribution": 0.15},
        ],
    ),
    NarrativeRequest(
        customer_id="CUST_002",
        churn_probability=0.72,
        risk_tier="high",
        top_shap_features=[
            {"feature": "monthly_charges", "contribution": 0.19},
            {"feature": "contract_type", "contribution": 0.16},
            {"feature": "tenure_months", "contribution": 0.12},
        ],
    ),
]

# Our feature names are not English. Without these definitions the model has to guess
# what "tenure_months" means, and a guess in the prompt becomes a guess in the narrative.
feature_defs = {
    "contract_type": "Whether the customer is on month-to-month, annual, or two-year plan",
    "support_tickets": "Number of times the customer contacted support",
    "tenure_months": "How long they've been a customer",
    "monthly_charges": "What they pay each month",
}

prompt = build_narrative_prompt(batch, feature_definitions=feature_defs)

print(f"One prompt covering {len(batch)} customers, {len(prompt)} characters:")
print("=" * 60)
print(prompt[:1200])
print("...")

In [ ]:
# Same shape as the mapping call: real when we can, saved when we can't.
#
# Note the reply format. Mapping asked for JSON; this asks for CUSTOMER_ID: / NARRATIVE:
# lines. That is deliberate. These paragraphs are full of quotes, dollar signs and
# apostrophes, and one unescaped character costs you the whole batch when you parse
# JSON. A line-prefixed format degrades instead: a mangled paragraph loses one customer
# and the other 49 still parse.
SAVED_NARRATIVE_RESPONSE = """CUSTOMER_ID: CUST_001
NARRATIVE: This customer is at very high risk of leaving. They have no long-term commitment (month-to-month plan), which means there's zero friction to cancel. They've also contacted support 5 times in just 2 months — a strong signal of frustration. With only 2 months of tenure, they haven't built any loyalty yet. Consider offering a discounted annual plan to lock them in, and escalate their open support issues immediately.

CUSTOMER_ID: CUST_002
NARRATIVE: This customer is paying significantly more than average ($110/month) on a month-to-month plan. High charges without a commitment create a "why am I paying this much?" moment. They're relatively new (4 months), so they're still in the window where switching costs are low. A loyalty discount or plan review could reduce their perceived cost and extend their stay."""

narratives = None
if LIVE:
    try:
        narratives = call_bedrock_for_narratives(prompt)
        print(f"Live from Bedrock: {len(narratives)} narratives.\n")
    except BedrockCallError as e:
        print(f"Bedrock call failed: {e}")
        print("Using the saved response so you can keep reading.\n")

if narratives is None:
    # The second argument is the guest list: any ID the reply does not mention gets a
    # logged warning, so a short batch announces itself the moment it happens.
    narratives = parse_narrative_response(SAVED_NARRATIVE_RESPONSE, ["CUST_001", "CUST_002"])

print("Generated narratives:")
print("=" * 60)
for cust_id, narrative in narratives.items():
    print(f"\n{cust_id}:")
    print(f"  {narrative[:200]}..." if len(narrative) > 200 else f"  {narrative}")

## What happens when it fails

This is the section the saved responses have been standing in for. On a real run
there is no saved reply: a failed call raises `BedrockCallError` with the cause in the
message, and what happens next depends on the step.

| Step | On failure | Who finds out |
|---|---|---|
| Auto-mapping | The call raises and there is no draft. | The operator running onboarding sees the error at once and writes the YAML by hand. Until then the client is simply not processed: `load_mapping_config` raises on a file that does not exist. |
| Narratives | Every failed customer is recorded with its reason and `"N/A"` in the output. | Under the failure budget, the failures ride along in the results for the run summary. Over it, the whole run raises `NarrativeGenerationError` and ships nothing. |

Two decisions in that table are doing quiet work.

**There is no default narrative.** A failed customer reads `"N/A"`, which is ugly on
purpose. A plausible sentence from a template would look exactly like something the
model wrote, and nobody downstream could tell explanation from filler.

**The budget is written down.** `generate_narratives_for_batch(..., failure_budget=0.10)`
compares every run against a number. One customer missing from a batch of 400 is a
recorded reason in the results. Two hundred missing is an outage, and the correct
output of an outage is an error, because a report that says `"N/A"` four hundred times
is how a client finds out before you do.

The cell below runs a real outage and shows both halves: the reasons recorded per
customer, and the run refusing to ship.

In [ ]:
# Stand in for Bedrock during an outage. A client whose calls raise the way boto3
# raises produces the same BedrockCallError a real outage would.
import botocore.exceptions


class UnreachableBedrock:
    def converse(self, **kwargs):
        raise botocore.exceptions.EndpointConnectionError(
            endpoint_url="https://bedrock-runtime.us-east-1.amazonaws.com"
        )


# Two customers, one batch, and the batch fails. That is a 100% failure rate, far over
# the 10% default budget, so the run raises instead of returning a table of N/A.
try:
    generate_narratives_for_batch(batch, boto3_client=UnreachableBedrock())
except NarrativeGenerationError as e:
    print(f"The run refused to ship:\n  {e}")
    print("\nWhat it recorded before stopping (e.results):")
    for cust_id, result in e.results.items():
        print(f"  {cust_id}: success={result.success}")
        print(f"     reason: {result.failure_reason}")

### What stage 10 still owes this

The counting happens in the library now, and this chapter stops there. Three pieces are
still manual: the failure rate belongs in the run summary next to the AUC, a raised
`NarrativeGenerationError` should page someone, and the reasons deserve different
responses, since throttling is worth a retry and an expired key never is. Stage 10
wires those up.

### The failure this chapter cannot catch

Everything above is the model failing to answer. The expensive failure is the model
answering, fluently, and being wrong.

A mapping that sends `complaints` to `support_tickets` when the client meant something
else does not raise. It renames the column, the row counts match, validation passes, and
the model trains on a feature that means something other than its name. You find out
when the predictions are quietly bad, months later, and the mapping is the last place
anyone looks.

No `try` block catches that. It is why Part 1 ends with a person renaming a file instead
of a confidence threshold. `confidence: high` is the model grading its own homework. The
rename is somebody signing it.

## The System Prompt

Look back at the prompt printed in Part 2. It contains the feature definitions, the
customers and the reply format, and none of the writing rules. Those travel separately,
in the system channel, and `call_bedrock_for_narratives` attaches them to every call.

**Why separate?** The system prompt is about *how to write*, and it is identical for
every customer, every batch and every client. The user message is the data, and it
changes every time. Splitting them keeps the instructions out of the part that varies.

The rules themselves are four, and they are blunt on purpose:

- **Non-technical language.** No "SHAP values", no "feature importance".
- **Under 150 words.** Long enough to explain, short enough to read before a call.
- **Reference specific values.** "5 support tickets", with the number in it.
- **Plain English.** A VP should understand every word.

**Part 1 has no system prompt at all.** Its instructions sit inline in the user message,
because a mapping prompt is sent once per client and there is nothing to keep constant
across calls. Two steps in the same chapter, two different answers, and the deciding
question is the same one: does this instruction change with the data, or not?

In [ ]:
print("System prompt used for narrative generation:")
print("=" * 60)
print(SYSTEM_PROMPT)

## Conclusions

This chapter added two LLM steps to the pipeline. The useful lesson is how differently
they had to be treated, and why.

Auto-mapping sits at onboarding. It reads a client's column names the way a person
would, which removes the typing from that job. The judgement stays. The draft it writes
cannot enter the pipeline until someone reviews it, edits it and renames it, and
`load_mapping_config` enforces the rule in code. All that ceremony exists because of
the one failure no error handler can see: a mapping that is wrong and plausible. Code
catches absent answers. Only a person catches confident ones.

Narrative generation sits at output, and it is the one genuinely optional step in the
pipeline. Its failure handling is accounting: every customer that comes back without a
narrative carries the reason, the run compares its failure rate to a written-down
budget, and past the budget it raises. The narratives themselves are shaped almost
entirely by the system prompt, four blunt rules that read like an editor's style sheet.

The economics work because of batching. Fifty customers share one call and one
preamble, and that is the whole difference between narratives costing cents per run
and narratives being a line item somebody questions.

Chapter 8 moves this pipeline onto SageMaker Pipelines, where the steps you have been
running by hand become a graph AWS runs for you.

---

*Source code: `src/churn_pipeline/llm/auto_mapping.py`, `src/churn_pipeline/llm/narrative_generator.py`*  
*Tests: `tests/unit/test_auto_mapping.py`, `tests/property/test_narrative.py`, `tests/unit/test_narrative.py`*  
*Series: [Build with AWS](https://buildwithaws.substack.com/)*